# Capital Rentals

MATH 153 ~ Group Project ~ Team Members: Karma, Melvin, Risav, Shirjak

In [1]:
import json, re, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "data/"

## Part 1. Concatenating datasets

### 1.1 – 1.3 Load, rename, convert, stack

In [2]:
q3 = pd.read_csv(DATA + "listings_q3.csv")
q4 = pd.read_csv(DATA + "listings_q4.csv")

# 1.1 rename Q4 columns to match Q3 exactly (positional order is identical, but we map by name to be explicit)
q4 = q4.rename(columns={
    "ListingID": "listing_id", "Landlord": "landlord_id", "Neighborhood": "nbhd",
    "Bedrooms": "beds", "Bathrooms": "baths", "SqFt": "sqft",
    "Monthly_Rent": "rent", "Date_Listed": "listed_on",
})
assert list(q3.columns) == list(q4.columns)

# 1.2 rent text -> float; dates -> datetime (explicit formats)
q4["rent"] = q4["rent"].str.replace(r"[$,]", "", regex=True).astype(float)
q3["listed_on"] = pd.to_datetime(q3["listed_on"], format="%Y-%m-%d")   # Q3 is ISO
q4["listed_on"] = pd.to_datetime(q4["listed_on"], format="%m/%d/%Y")   # Q4 is MM/DD/YYYY

# 1.3 source column + stack
q3["source"] = "q3"
q4["source"] = "q4"
stacked = pd.concat([q3, q4], ignore_index=True)
print("q3:", q3.shape, "| q4:", q4.shape, "| stacked:", stacked.shape)
stacked.dtypes

q3: (180, 9) | q4: (126, 9) | stacked: (306, 9)


listing_id             object
landlord_id            object
nbhd                   object
beds                    int64
baths                 float64
sqft                  float64
rent                  float64
listed_on      datetime64[ns]
source                 object
dtype: object

**Formats used:** Q3 dates format="%Y-%m-%d", Q4 dates format="%m/%d/%Y". Rent in Q4 was text like `"$4,290"`; we strip `$` and `,` with a regex, then cast to float.

### 1.4 Duplicates (re-posted listings)

In [5]:
dups = stacked[stacked["listing_id"].duplicated(keep=False)].sort_values(["listing_id", "source"])
display(dups)

# To find if the two copies identical apart from 'source'?
cols = [c for c in stacked.columns if c != "source"]
print("Re-posted IDs:", dups["listing_id"].nunique())
print("Copies identical on all other columns:", dups.groupby("listing_id")[cols].nunique().max().max() == 1)

before = len(stacked)
# keep the Q4 copy (the most recent posting). Since copies are identical it makes no numeric difference,
# but the latest posting is the right default if they ever differed.
listings = (stacked.sort_values(["listing_id", "listed_on", "source"])
                   .drop_duplicates("listing_id", keep="last")
                   .sort_values("listing_id").reset_index(drop=True))
print(f"rows before: {before} | rows after: {len(listings)} | removed: {before - len(listings)}")

,listing_id,landlord_id,nbhd,beds,baths,sqft,rent,listed_on,source
36,R-10037,LL123,N11,0,1.0,340.0,1560.0,2026-07-14,q3
303,R-10037,LL123,N11,0,1.0,340.0,1560.0,2026-07-14,q4
126,R-10127,LL121,Nbhd 08,1,1.0,650.0,2740.0,2026-09-18,q3
271,R-10127,LL121,Nbhd 08,1,1.0,650.0,2740.0,2026-09-18,q4
133,R-10134,LL106,n-05,1,1.0,690.0,2430.0,2026-09-18,q3
219,R-10134,LL106,n-05,1,1.0,690.0,2430.0,2026-09-18,q4
138,R-10139,LL114,N04,1,1.0,870.0,3040.0,2026-09-18,q3
251,R-10139,LL114,N04,1,1.0,870.0,3040.0,2026-09-18,q4
143,R-10144,LL109,n10,4,2.0,1720.0,5990.0,2026-09-20,q3
252,R-10144,LL109,n10,4,2.0,1720.0,5990.0,2026-09-20,q4


Re-posted IDs: 6
Copies identical on all other columns: True
rows before: 306 | rows after: 300 | removed: 6


**Finding:** Every re-posted listing is an exact copy (same landlord, rent, date), so which one we keep does not change any number. We keep the later (Q4) copy by convention. Leaving duplicates in would double-count those listings in every average and in the SQL counts.

## Part 2. Regular expressions

### 2.1 Cleaning the neighborhood code

In [8]:
spellings = sorted(listings["nbhd"].unique(), key=lambda s: (s.strip().lower().replace("-", "").replace(" ", "")))
print(len(spellings), "distinct raw spellings:")
print([repr(s) for s in spellings])

59 distinct raw spellings:
["'N01'", "'n01'", "'n-01'", "'N02'", "'n02'", "'n-02'", "'N03'", "'n-03'", "'n03'", "'N04'", "'n-04'", "'n04'", "'n05'", "'N05'", "'n-05'", "'N06'", "'n-06'", "'N07'", "'n07'", "'n-07'", "'n08'", "'N08'", "'n-08'", "'n09'", "'n-09'", "'N09'", "' N1 '", "'n-10'", "' N10 '", "'N10'", "'n10'", "'n-11'", "'N11'", "' N11 '", "'n11'", "'n12'", "'N12'", "'n-12'", "' N12 '", "' N2 '", "' N3 '", "' N4 '", "' N5 '", "' N6 '", "' N7 '", "' N8 '", "' N9 '", "'Nbhd 01'", "'Nbhd 02'", "'Nbhd 03'", "'Nbhd 04'", "'Nbhd 05'", "'Nbhd 06'", "'Nbhd 07'", "'Nbhd 08'", "'Nbhd 09'", "'Nbhd 10'", "'Nbhd 11'", "'Nbhd 12'"]


In [9]:
# One regex: optional spaces, N/n, optional 'bhd', optional space/hyphen, optional leading 0, 1-2 digits, optional spaces.
NBHD_RE = r"^\s*[Nn](?:bhd)?[\s-]*0?(\d{1,2})\s*$"
listings["nbhd_code"] = listings["nbhd"].str.replace(NBHD_RE, lambda m: f"N{int(m.group(1)):02d}", regex=True)
codes = sorted(listings["nbhd_code"].unique())
print(len(codes), codes)
assert len(codes) == 12 and all(re.fullmatch(r"N(0[1-9]|1[0-2])", c) for c in codes)

12 ['N01', 'N02', 'N03', 'N04', 'N05', 'N06', 'N07', 'N08', 'N09', 'N10', 'N11', 'N12']


**Pattern explained:** `^\s*` allows leading spaces; `[Nn]` the letter; `(?:bhd)?` the optional word "bhd" (non-capturing); `[\s-]*` any mix of spaces/hyphens; `0?` an optional leading zero; `(\d{1,2})` captures the number (1 or 2 digits); `\s*$` trailing spaces. The replacement function re-pads the captured number to two digits and puts `N` in front. The assertion proves exactly 12 valid codes (N01–N12) remain.

### 2.2 Parsing raw_ads.txt

In [10]:
PATTERNS = {
    # R- followed by exactly five digits
    "listing_id": r"R-\d{5}",
    # a number (with optional thousands commas) followed by sq ft / sqft / sq.ft. / SF  (case-insensitive)
    "sqft": r"(\d{1,3}(?:,\d{3})+|\d+)\s*(?:sq\.?\s*ft\.?|sqft|SF)\b",
    # dollar sign then digits with optional thousands commas
    "rent": r"\$(\d{1,3}(?:,\d{3})+|\d+)",
    # (ddd) ddd-dddd
    "phone": r"\(\d{3}\)\s*\d{3}-\d{4}",
    # local@domain.tld
    "email": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)*\.[a-z]{2,}",
}

def first(pattern, line, group=0):
    m = re.search(pattern, line, flags=re.IGNORECASE)
    return m.group(group) if m else None

rows = []
with open(DATA + "raw_ads.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        sq, rent = first(PATTERNS["sqft"], line, 1), first(PATTERNS["rent"], line, 1)
        rows.append({
            "listing_id": first(PATTERNS["listing_id"], line),
            "ad_sqft": float(sq.replace(",", "")) if sq else np.nan,
            "ad_rent": float(rent.replace(",", "")) if rent else np.nan,
            "ad_phone": first(PATTERNS["phone"], line),
            "ad_email": first(PATTERNS["email"], line),
        })
ads = pd.DataFrame(rows)
print(ads.shape)
display(ads.head(10))
print(ads.isna().sum())

(32, 5)


,listing_id,ad_sqft,ad_rent,ad_phone,ad_email
0,R-10274,1880.0,6490.0,(301) 555-1370,potomacpropertygroup@rentals.net
1,R-10140,1170.0,2120.0,(202) 555-1185,anacostiahomes@rentals.net
2,R-10155,770.0,2610.0,(240) 555-1407,luis.ortiz@mail.com
3,R-10275,360.0,1520.0,None,hassan.karim@mail.com
4,R-10247,1260.0,5910.0,(301) 555-1555,capitolkeys@rentals.net
5,R-10291,780.0,2930.0,(301) 555-1814,None
6,R-10235,1690.0,6850.0,None,northstarrentals@rentals.net
7,R-10128,1440.0,2840.0,(240) 555-1629,rosa.alvarez@mail.com
8,R-10111,1750.0,3410.0,(301) 555-1222,gloria.diallo@mail.com
9,R-10182,1200.0,2760.0,(301) 555-1333,jamal.wright@mail.com


listing_id    0
ad_sqft       0
ad_rent       0
ad_phone      6
ad_email      9
dtype: int64


**What each pattern does**
- `listing_id` `R-\d{5}`: literal `R-` and exactly five digits.
- `sqft` `(\d{1,3}(?:,\d{3})+|\d+)\s*(?:sq\.?\s*ft\.?|sqft|SF)\b`: captures either a comma-grouped number (`1,260`) or a plain number, then optional space, then any of `sq ft`, `sq.ft.`, `sqft`, `SF`. Case-insensitive; `\b` stops it matching inside longer words.
- `rent` `\$(...)`: the number after a dollar sign, same comma handling. Works for `$6,490 per month`, `$1,520/month`, `$2,930.`
- `phone` `\(\d{3}\)\s*\d{3}-\d{4}`: area code in literal parentheses.
- `email`: word characters/dots before `@`, domain labels, and a final TLD of 2+ letters. Because the TLD must be letters, a sentence-ending period (`...mail.com.`) is not swallowed.

Phone and email are legitimately missing in some ads (some give only one contact method), so those NaNs are expected.

### 2.3 Merge ads onto listings, check rent, fill sqft

In [11]:
merged = listings.merge(ads, on="listing_id", how="outer", indicator=True)
print(merged["_merge"].value_counts())

both = merged[merged["_merge"] == "both"]
print("\nmatched rows:", len(both))
print("rent disagreements (ad_rent vs rent):", (both["ad_rent"] != both["rent"]).sum())
print("ads that had a sqft where listing ALSO had a sqft:", both["sqft"].notna().sum(),
      "| of those, conflicting:", (both["sqft"].notna() & (both["sqft"] != both["ad_sqft"])).sum())

missing_before = listings["sqft"].isna().sum()
listings = listings.merge(ads, on="listing_id", how="left")
listings["sqft"] = listings["sqft"].fillna(listings["ad_sqft"])
missing_after = listings["sqft"].isna().sum()
print(f"\nsqft missing before: {missing_before} | after: {missing_after}")

_merge
left_only     268
both           32
right_only      0
Name: count, dtype: int64

matched rows: 32
rent disagreements (ad_rent vs rent): 0
ads that had a sqft where listing ALSO had a sqft: 0 | of those, conflicting: 0

sqft missing before: 32 | after: 0


**Finding:** The ads agree with the CSV rent on every matched row, so they are a trustworthy second source. They exist exactly for the listings that have no square footage in the CSVs, so `fillna` with `ad_sqft` fills every gap.